In [16]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import copy
import os
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

SEED = 42
VAL_FRACTION = 0.15
N_FOLDS = 5
BATCH_SIZE = 32
EPOCHS = 25
LR = 1e-3

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Get images ---
# Note: Standard image size is 640 x 640
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

full_dataset = datasets.ImageFolder(root="Dataset/")
print("Detected classes:", full_dataset.class_to_idx)

# --- Load folds.csv & Split ---
df = pd.read_csv("folds.csv")


def get_split(df, k):
  test = df[df["fold"] == k]
  rest = df[df["fold"] != k]
  train, val = train_test_split(
      rest, test_size=VAL_FRACTION, stratify=rest["class_id"], random_state=SEED
  )
  return train, val, test


def make_dataset(base_dataset, split_df, transform):
  ds = copy.copy(base_dataset)
  ds.transform = transform
  ds.samples = [
      (os.path.normpath(p), int(c))
      for p, c in zip(split_df["path"], split_df["class_id"])
  ]
  ds.targets = [c for _, c in ds.samples]
  return ds


# Pick fold round k = 0
train_df, val_df, test_df = get_split(df, k=0)

train_dataset = make_dataset(full_dataset, train_df, train_transform)
val_dataset = make_dataset(full_dataset, val_df, val_test_transform)
test_dataset = make_dataset(full_dataset, test_df, val_test_transform)

print(
    f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test:"
    f" {len(test_dataset)}"
)

dataloader_train = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dataloader_val = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
dataloader_test = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


Detected classes: {'Crack': 0, 'Pothole': 1, 'Surface Erosion': 2}
Train: 853, Val: 151, Test: 252


### Make CNN

In [17]:
from torch import nn
import torch.nn.functional as F


# --- Make CNN ---

class CNN(nn.Module):
    def __init__(self, in_channels=3, num_classes=3):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=8, kernel_size=7, padding=3)

        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=5, padding=2)

        self.conv3 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((7, 7))
        self.fc1 = nn.Linear(32 * 7 * 7, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.adaptive_pool(x)
        x = x.reshape(x.shape[0], -1)
        x = self.fc1(x)
        return x




### Training

In [18]:
from torch import optim

# --- Train CNN ---

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = CNN(in_channels=3, num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)#
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

best_val_loss = float("inf")
best_weights = None

for epoch in range(1, EPOCHS + 1):

  # Train on train dataset
  model.train()
  train_loss = 0.0
  for images, labels in dataloader_train:
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()
    loss = criterion(model(images), labels)
    loss.backward()
    optimizer.step()
    train_loss += loss.item() * images.size(0)

  # Evaluate on validaiton dataset (to adjust hyperparameters)
  model.eval()
  val_loss, val_correct = 0.0, 0
  with torch.no_grad():
    for images, labels in dataloader_val:
      images, labels = images.to(device), labels.to(device)
      outputs = model(images)
      val_loss += criterion(outputs, labels).item() * images.size(0)
      val_correct += (outputs.argmax(dim=1) == labels).sum().item()

  avg_train_loss = train_loss / len(train_dataset)
  avg_val_loss = val_loss / len(val_dataset)
  val_acc = val_correct / len(val_dataset)

  # Adjust LR based on validation loss
  scheduler.step(avg_val_loss)
  current_lr = optimizer.param_groups[0]["lr"]

  # Save best
  if avg_val_loss < best_val_loss:
    best_val_loss = avg_val_loss
    best_weights = copy.deepcopy(model.state_dict())

  print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | LR: {current_lr:.6f}")


Device: cpu
Epoch 01/25 | Train Loss: 1.0140 | Val Loss: 0.8541 | Val Acc: 0.6159 | LR: 0.001000
Epoch 02/25 | Train Loss: 0.9102 | Val Loss: 0.8657 | Val Acc: 0.5960 | LR: 0.001000
Epoch 03/25 | Train Loss: 0.8628 | Val Loss: 0.8578 | Val Acc: 0.6159 | LR: 0.001000
Epoch 04/25 | Train Loss: 0.8100 | Val Loss: 0.7866 | Val Acc: 0.6821 | LR: 0.001000
Epoch 05/25 | Train Loss: 0.7777 | Val Loss: 0.8160 | Val Acc: 0.6755 | LR: 0.001000
Epoch 06/25 | Train Loss: 0.7554 | Val Loss: 0.8136 | Val Acc: 0.6424 | LR: 0.001000
Epoch 07/25 | Train Loss: 0.7456 | Val Loss: 0.7552 | Val Acc: 0.6954 | LR: 0.001000
Epoch 08/25 | Train Loss: 0.6835 | Val Loss: 0.7135 | Val Acc: 0.7152 | LR: 0.001000
Epoch 09/25 | Train Loss: 0.7140 | Val Loss: 0.6836 | Val Acc: 0.7417 | LR: 0.001000
Epoch 10/25 | Train Loss: 0.6669 | Val Loss: 0.7521 | Val Acc: 0.6887 | LR: 0.001000
Epoch 11/25 | Train Loss: 0.6579 | Val Loss: 0.6924 | Val Acc: 0.7086 | LR: 0.001000
Epoch 12/25 | Train Loss: 0.6447 | Val Loss: 0.6826 |

### Evaluation

In [19]:
from torchmetrics import Accuracy, Precision, Recall

# --- Evaluate CNN ---
acc = Accuracy(task="multiclass",num_classes=3)
precision = Precision(task="multiclass", num_classes=3, average="macro")
recall = Recall(task="multiclass", num_classes=3, average="macro")

model.eval()
with torch.no_grad():
    for images, labels in dataloader_test:
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        
        # Update
        acc(preds, labels)
        precision(preds, labels)
        recall(preds, labels)

print(f"Test Accuracy:  {acc.compute():.4f}")
print(f"Test Precision: {precision.compute():.4f}")
print(f"Test Recall:    {recall.compute():.4f}")

Test Accuracy:  0.8175
Test Precision: 0.8201
Test Recall:    0.8235
